In [0]:
# DBTITLE 1, Integration to Final + Preferred All Rebuild
# ============================================================
# 목적
# 1. sandbox.z_jungryo_lee.category_topic_detail_integration 기준으로
#    sandbox.t_online_voc_analysis.voc_llm_topic_classification 업데이트
#
# 2. memo_id 보정 로직 기반으로
#    sandbox.z_jungryo_lee.voc_llm_topic_classification_preferred_all 전체 재생성
#
# 참고 로직
# - round8 파일: integration merge / final 적재 구조
# - memo_id 보정 파일: memo_id repair + preferred_all overwrite 구조
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
import uuid

# ------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------
SAVE_DB = "sandbox.z_jungryo_lee"

INTEGRATION_TABLE = f"{SAVE_DB}.category_topic_detail_integration"
FINAL_OUTPUT_TABLE = "sandbox.t_online_voc_analysis.voc_llm_topic_classification"
PREFERRED_TABLE = f"{SAVE_DB}.voc_llm_topic_classification_preferred_all"

MEMO_MAP_TABLE = f"{SAVE_DB}.buzzmetrix_memo_id_map_all"

TARGET_CATE_1 = "07. 스마트 사용성"

print("[CONFIG]")
print(f"  INTEGRATION_TABLE  = {INTEGRATION_TABLE}")
print(f"  FINAL_OUTPUT_TABLE = {FINAL_OUTPUT_TABLE}")
print(f"  PREFERRED_TABLE    = {PREFERRED_TABLE}")
print(f"  MEMO_MAP_TABLE     = {MEMO_MAP_TABLE}")


# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------
def normalize_memo_expr(col_name: str):
    return F.trim(
        F.regexp_replace(
            F.translate(F.col(col_name).cast("string"), "　", " "),
            "\\s+",
            " ",
        )
    )


def with_memo_id(df):
    return df.withColumn(
        "memo_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("cate_1_depth").cast("string"), F.lit("")),
                F.coalesce(F.col("cate_2_depth").cast("string"), F.lit("")),
                F.coalesce(F.col("sc_measurement").cast("string"), F.lit("")),
                normalize_memo_expr("memo"),
            ),
            256,
        ),
    )


def ensure_column(table_name: str, column_name: str, column_type: str = "STRING"):
    if column_name not in spark.table(table_name).columns:
        spark.sql(f"ALTER TABLE {table_name} ADD COLUMN {column_name} {column_type}")
        print(f"  [ALTER] {table_name} + {column_name}")


def align_to_target_columns(df, table_name: str):
    target_cols = spark.table(table_name).columns

    for c in target_cols:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast("string"))

    return df.select(*target_cols)


def extract_round_num_expr(col_name: str):
    return F.when(
        F.regexp_extract(F.lower(F.col(col_name)), r"(\d+)", 1) != "",
        F.regexp_extract(F.lower(F.col(col_name)), r"(\d+)", 1).cast("int"),
    )


def dedupe_by_keys(df, key_columns):
    order_exprs = []

    for c in [
        "topic_rev",
        "main_topic",
        "llm_round",
        "topic",
        "description",
        "data_created_dt",
        "_row_id",
    ]:
        if c in df.columns:
            order_exprs.append(F.col(c).isNotNull().cast("int").desc())
            order_exprs.append(F.col(c).desc_nulls_last())

    if not order_exprs:
        order_exprs.append(F.lit(1))

    before_cnt = df.count()

    w = Window.partitionBy(*key_columns).orderBy(*order_exprs)

    deduped_df = (
        df.withColumn("_merge_rn", F.row_number().over(w))
        .where(F.col("_merge_rn") == 1)
        .drop("_merge_rn")
    )

    after_cnt = deduped_df.count()

    if before_cnt != after_cnt:
        print(f"  [DEDUP] {key_columns}: {before_cnt:,} -> {after_cnt:,}")

    return deduped_df


def merge_table_by_keys(df, table_name: str, key_columns):
    row_cnt = df.count()

    if row_cnt == 0:
        print(f"  [SKIP MERGE] {table_name}, keys={key_columns}, rows=0")
        return

    df = dedupe_by_keys(df, key_columns)

    temp_view = f"tmp_merge_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(temp_view)

    on_clause = " AND ".join([f"t.`{c}` <=> s.`{c}`" for c in key_columns])

    spark.sql(f"""
        MERGE INTO {table_name} t
        USING {temp_view} s
        ON {on_clause}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    spark.catalog.dropTempView(temp_view)


def safe_merge_table(df, table_name: str):
    """
    OR 조건 MERGE 금지.
    1. memo_id 있는 row는 memo_id 기준 merge
    2. memo_id 없는 row만 memo + category + sc 기준 merge
    """
    df = align_to_target_columns(df, table_name)

    memo_id_df = df.where(F.col("memo_id").isNotNull())
    legacy_df = df.where(F.col("memo_id").isNull())

    memo_id_cnt = memo_id_df.count()
    legacy_cnt = legacy_df.count()

    print(f"  memo_id merge 대상 : {memo_id_cnt:,}")
    print(f"  legacy merge 대상  : {legacy_cnt:,}")

    if memo_id_cnt > 0:
        merge_table_by_keys(memo_id_df, table_name, ["memo_id"])

    if legacy_cnt > 0:
        legacy_keys = ["memo", "cate_1_depth", "cate_2_depth", "sc_measurement"]
        merge_table_by_keys(legacy_df, table_name, legacy_keys)


def repair_df_with_memo_map(df, memo_map_df):
    """
    기존 memo_id map 기반 memo / memo_id 보정.
    map에 없는 경우 동일 산식으로 memo_id fallback 생성.
    """
    if "memo_id" not in df.columns:
        df = df.withColumn("memo_id", F.lit(None).cast("string"))

    target_cols = df.columns

    repaired_df = (
        df.withColumn("memo_norm", normalize_memo_expr("memo"))
        .alias("t")
        .join(
            memo_map_df.alias("m"),
            on=["cate_1_depth", "cate_2_depth", "sc_measurement", "memo_norm"],
            how="left",
        )
        .select(
            *[
                F.col(f"t.{c}").alias(c)
                for c in target_cols
                if c not in {"memo", "memo_id"}
            ],
            F.coalesce(F.col("m.source_memo"), F.col("t.memo")).alias("memo"),
            F.coalesce(
                F.col("t.memo_id"),
                F.col("m.memo_id"),
                F.sha2(
                    F.concat_ws(
                        "||",
                        F.coalesce(F.col("t.cate_1_depth").cast("string"), F.lit("")),
                        F.coalesce(F.col("t.cate_2_depth").cast("string"), F.lit("")),
                        F.coalesce(F.col("t.sc_measurement").cast("string"), F.lit("")),
                        F.coalesce(F.col("t.memo_norm"), F.lit("")),
                    ),
                    256,
                ),
            ).alias("memo_id"),
        )
    )

    return repaired_df


# ------------------------------------------------------------
# 2. 기존 memo_id map 로드
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[STEP 1] 기존 memo_id map 로드")
print("=" * 80)

if not spark.catalog.tableExists(MEMO_MAP_TABLE):
    raise RuntimeError(f"[ERROR] memo_id map 테이블 없음: {MEMO_MAP_TABLE}")

memo_map_df = spark.table(MEMO_MAP_TABLE)

required_map_cols = {
    "cate_1_depth",
    "cate_2_depth",
    "sc_measurement",
    "memo_norm",
    "source_memo",
    "memo_id",
}

missing_cols = required_map_cols - set(memo_map_df.columns)

if missing_cols:
    raise RuntimeError(f"[ERROR] memo_id map 필수 컬럼 누락: {missing_cols}")

print(f"[LOAD] {MEMO_MAP_TABLE}: {memo_map_df.count():,}건")


# ------------------------------------------------------------
# 3. Integration -> Final 업데이트
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[STEP 2] Integration 기준 Final 업데이트")
print("=" * 80)

if not spark.catalog.tableExists(INTEGRATION_TABLE):
    raise RuntimeError(f"[ERROR] integration 테이블 없음: {INTEGRATION_TABLE}")

integration_df = spark.table(INTEGRATION_TABLE)

print(f"[LOAD] integration rows: {integration_df.count():,}")

# memo_summary는 final에 적재하지 않음
if "memo_summary" in integration_df.columns:
    integration_df = integration_df.drop("memo_summary")

# memo_id 보정
integration_df = repair_df_with_memo_map(integration_df, memo_map_df)

# final용 data_created_dt
integration_df = integration_df.withColumn("data_created_dt", F.current_date())

final_columns = [
    "cate_1_depth",
    "cate_2_depth",
    "sc_measurement",
    "topic",
    "_row_id",
    "memo_id",
    "memo",
    "description",
    "topic_rev",
    "main_topic",
    "llm_round",
    "data_created_dt",
]

integration_for_final_df = integration_df.select(
    *[c for c in final_columns if c in integration_df.columns]
)

print(f"  final 반영 후보: {integration_for_final_df.count():,}")

if spark.catalog.tableExists(FINAL_OUTPUT_TABLE):
    ensure_column(FINAL_OUTPUT_TABLE, "memo_id")

    before_cnt = spark.table(FINAL_OUTPUT_TABLE).count()
    print(f"  final 기존 rows: {before_cnt:,}")

    safe_merge_table(integration_for_final_df, FINAL_OUTPUT_TABLE)

    after_cnt = spark.table(FINAL_OUTPUT_TABLE).count()
    print(f"  [MERGE] final: {before_cnt:,} -> {after_cnt:,} ({after_cnt - before_cnt:+,})")

else:
    integration_for_final_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(FINAL_OUTPUT_TABLE)
    print(f"  [CREATE] final 생성: {integration_for_final_df.count():,}건")


# ------------------------------------------------------------
# 4. Final table memo_id repair overwrite
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[STEP 3] Final 전체 memo_id repair")
print("=" * 80)

final_df = spark.table(FINAL_OUTPUT_TABLE)

before_cnt = final_df.count()
null_before = final_df.where(
    F.col("memo").isNotNull() & F.col("memo_id").isNull()
).count()

final_repaired_df = repair_df_with_memo_map(final_df, memo_map_df)

null_after = final_repaired_df.where(
    F.col("memo").isNotNull() & F.col("memo_id").isNull()
).count()

final_repaired_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(FINAL_OUTPUT_TABLE)

print(f"[SAVE] {FINAL_OUTPUT_TABLE}")
print(f"  rows            : {before_cnt:,}")
print(f"  memo_id NULL 전 : {null_before:,}")
print(f"  memo_id NULL 후 : {null_after:,}")


# ------------------------------------------------------------
# 5. preferred_all 전체 재생성
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[STEP 4] preferred_all 전체 재생성")
print("=" * 80)

classified_repaired_df = (
    spark.table(FINAL_OUTPUT_TABLE)
    .where(F.col("memo").isNotNull())
    .withColumn("memo_norm", normalize_memo_expr("memo"))
    .transform(with_memo_id)
    .withColumn("llm_round_num", extract_round_num_expr("llm_round"))
)

print(f"[LOAD] final repaired rows: {classified_repaired_df.count():,}")

# 중복 memo_id 현황
overlap_stats_df = (
    classified_repaired_df
    .groupBy("memo_id")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("llm_round").alias("llm_round_count"),
    )
    .where(F.col("row_count") > 1)
)

print(f"[OVERLAP] duplicated memo_id count: {overlap_stats_df.count():,}")

display(
    overlap_stats_df
    .orderBy(F.desc("row_count"), F.desc("llm_round_count"))
    .limit(20)
)

# 기존 memo_id 보정 파일의 preferred 우선순위 유지
preferred_window = Window.partitionBy("memo_id").orderBy(
    F.when(F.col("llm_round_num") == 2, F.lit(0))
     .when(F.col("llm_round_num") == 1, F.lit(1))
     .when(F.col("llm_round_num") == 0, F.lit(2))
     .otherwise(F.lit(3)).asc(),
    F.when(F.col("llm_round_num").isin(2, 1, 0), F.col("llm_round_num")).desc_nulls_last(),
    F.when(~F.col("llm_round_num").isin(2, 1, 0), F.col("llm_round_num")).desc_nulls_last(),
    F.col("data_created_dt").desc_nulls_last(),
    F.col("llm_round").desc_nulls_last(),
)

classified_preferred_df = (
    classified_repaired_df
    .withColumn("memo_pick_rank", F.row_number().over(preferred_window))
    .where(F.col("memo_pick_rank") == 1)
    .drop("memo_pick_rank", "llm_round_num", "memo_norm", "_row_id")
)

# topic fallback
fallback_topic_df = (
    classified_repaired_df
    .where(F.col("topic").isNotNull() & (F.length(F.trim(F.col("topic"))) > 0))
    .withColumn("topic_fallback_rank", F.row_number().over(preferred_window))
    .where(F.col("topic_fallback_rank") == 1)
    .select(
        "memo_id",
        F.col("topic").alias("fallback_topic"),
    )
)

topic_missing_condition = (
    F.col("p.topic").isNull()
    | (F.length(F.trim(F.col("p.topic"))) == 0)
)

classified_preferred_filled_df = (
    classified_preferred_df.alias("p")
    .join(fallback_topic_df.alias("f"), on="memo_id", how="left")
    .withColumn(
        "topic",
        F.when(topic_missing_condition, F.col("f.fallback_topic"))
         .otherwise(F.col("p.topic"))
    )
    .drop("fallback_topic")
)

filled_topic_count = (
    classified_preferred_df.alias("p")
    .join(fallback_topic_df.alias("f"), on="memo_id", how="inner")
    .where(topic_missing_condition & F.col("f.fallback_topic").isNotNull())
    .count()
)

print(f"  preferred rows       : {classified_preferred_filled_df.count():,}")
print(f"  topic fallback count : {filled_topic_count:,}")

classified_preferred_filled_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(PREFERRED_TABLE)

print(f"[SAVE] {PREFERRED_TABLE}: {classified_preferred_filled_df.count():,}건")


# ------------------------------------------------------------
# 6. 검증
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("[STEP 5] 검증")
print("=" * 80)

print("\n[Final by llm_round]")
display(
    spark.table(FINAL_OUTPUT_TABLE)
    .groupBy("llm_round")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("memo_id").alias("memo_id_count"),
        F.sum(F.when(F.col("memo_id").isNull(), 1).otherwise(0)).alias("memo_id_null_count"),
        F.sum(F.when(F.col("topic_rev").isNull(), 1).otherwise(0)).alias("topic_rev_null_count"),
        F.min("data_created_dt").alias("min_dt"),
        F.max("data_created_dt").alias("max_dt"),
    )
    .orderBy("llm_round")
)

print("\n[Preferred by llm_round]")
display(
    spark.table(PREFERRED_TABLE)
    .groupBy("llm_round")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("memo_id").alias("memo_id_count"),
        F.sum(F.when(F.col("memo_id").isNull(), 1).otherwise(0)).alias("memo_id_null_count"),
        F.sum(
            F.when(
                F.col("topic").isNull() | (F.length(F.trim(F.col("topic"))) == 0),
                1,
            ).otherwise(0)
        ).alias("topic_null_count"),
        F.sum(F.when(F.col("topic_rev").isNull(), 1).otherwise(0)).alias("topic_rev_null_count"),
    )
    .orderBy("llm_round")
)

print("\n[Final r10/r11]")
display(
    spark.table(FINAL_OUTPUT_TABLE)
    .where(F.col("llm_round").isin("r10", "r11"))
    .groupBy("llm_round")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("memo_id").alias("memo_id_count"),
    )
    .orderBy("llm_round")
)

print("\n[Preferred r10/r11]")
display(
    spark.table(PREFERRED_TABLE)
    .where(F.col("llm_round").isin("r10", "r11"))
    .groupBy("llm_round")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("memo_id").alias("memo_id_count"),
    )
    .orderBy("llm_round")
)

print("\n✅ integration 기반 final 업데이트 + preferred_all 전체 재생성 완료")